In [1]:
from pathlib import Path
from datetime import datetime
import json
import re
import sys
import subprocess
import time
import zipfile

import pandas as pd

direktori_aktif = Path.cwd()

if direktori_aktif.name.lower() == "notebooks":
    direktori_project = direktori_aktif.parent
else:
    direktori_project = direktori_aktif

direktori_src = direktori_project / "src"
direktori_models = direktori_project / "models"
direktori_outputs = direktori_project / "reports" / "outputs"
direktori_examples = direktori_project / "examples"
direktori_samples = direktori_project / "data" / "samples_metadata" / "file_samples"

for folder in [direktori_src, direktori_models, direktori_outputs, direktori_examples]:
    folder.mkdir(parents=True, exist_ok=True)

file_wajib = {
    "engine_v5": direktori_src / "phishrisk_engine_v5.py",
    "cli_v5": direktori_src / "run_phishrisk_v5.py",
    "engine_v4": direktori_src / "phishrisk_engine_v4.py",
    "public_ti": direktori_src / "public_threat_intelligence.py",
    "file_analyzer": direktori_src / "file_static_analyzer.py",
    "model_best_v5": direktori_models / "model_terbaik_multi_dataset_v5.pkl",
    "model_xgb_v5": direktori_models / "model_xgb_multi_dataset_v5.pkl",
    "fitur_v5": direktori_outputs / "daftar_fitur_multi_dataset_v5.json",
}

validasi_awal = pd.DataFrame([
    {
        "nama_file": nama,
        "lokasi": str(lokasi),
        "tersedia": lokasi.exists(),
        "ukuran_mb": round(lokasi.stat().st_size / (1024 * 1024), 2) if lokasi.exists() else 0,
    }
    for nama, lokasi in file_wajib.items()
])

display(validasi_awal)

if not validasi_awal["tersedia"].all():
    raise FileNotFoundError("Ada file wajib STEP 17 yang belum tersedia.")

print("Semua file wajib STEP 17 tersedia.")


,nama_file,lokasi,tersedia,ukuran_mb
0,engine_v5,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v5.py,True,0.02
1,cli_v5,C:\Users\ASUS\PHISHING\src\run_phishrisk_v5.py,True,0.00
2,engine_v4,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v4.py,True,0.00
3,public_ti,C:\Users\ASUS\PHISHING\src\public_threat_intel...,True,0.01
4,file_analyzer,C:\Users\ASUS\PHISHING\src\file_static_analyze...,True,0.02
5,model_best_v5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,True,584.68
6,model_xgb_v5,C:\Users\ASUS\PHISHING\models\model_xgb_multi_...,True,1.89
7,fitur_v5,C:\Users\ASUS\PHISHING\reports\outputs\daftar_...,True,0.00


Semua file wajib STEP 17 tersedia.


In [2]:
if str(direktori_src) not in sys.path:
    sys.path.insert(0, str(direktori_src))

import importlib
import phishrisk_engine_v5

importlib.reload(phishrisk_engine_v5)

engine_best = phishrisk_engine_v5.PhishRiskEngineV5(
    direktori_project=direktori_project,
    prefer_model="best",
)

engine_xgb = phishrisk_engine_v5.PhishRiskEngineV5(
    direktori_project=direktori_project,
    prefer_model="xgb",
)

print("Engine V5 best siap.")
print("Model best:", engine_best.model_path)
print("Model xgb:", engine_xgb.model_path)


Engine V5 best siap.
Model best: C:\Users\ASUS\PHISHING\models\model_terbaik_multi_dataset_v5.pkl
Model xgb: C:\Users\ASUS\PHISHING\models\model_xgb_multi_dataset_v5.pkl


In [3]:
url_resmi = [
    "https://praktikum.gunadarma.ac.id",
    "https://baak.gunadarma.ac.id",
    "https://www.bca.co.id",
    "https://www.shopee.co.id",
    "https://www.microsoft.com",
    "https://google.com",
    "https://youtube.com",
    "https://apple.com",
    "https://cloudflare.com",
    "https://github.com",
    "https://openai.com",
    "https://kompas.com",
    "https://detik.com",
    "https://tokopedia.com",
    "https://bukalapak.com",
]

url_berisiko = [
    "http://rricrosoft.com",
    "http://rnicrosoft.com",
    "http://micros0ft-login-update.test",
    "http://bca-login-update.test",
    "http://paypal-verify-account.test",
    "http://praktikum-gunadarma-login-update.test",
    "https://xn--micrsoft-q4a.test",
    "http://155.94.163.206/ai/?authenticated=true&account=login",
    "http://shopee-claim-reward.test",
    "http://google-security-update.test",
    "http://m1crosoft-account-verify.test",
    "http://bankbca-login-secure.test",
    "http://ovo-login-update.test",
    "http://dana-verify-account.test",
    "http://whatsapp-web-login-update.test",
]

url_netral = [
    "https://example.com",
    "https://example.org/docs",
    "https://wikipedia.org",
    "https://python.org",
    "https://pandas.pydata.org",
    "https://scikit-learn.org",
    "https://huggingface.co",
    "https://kaggle.com",
]

data_skenario_url = pd.DataFrame(
    [{"url": url, "skenario": "resmi", "expected": "Terlihat Aman"} for url in url_resmi]
    + [{"url": url, "skenario": "berisiko", "expected": "Berisiko"} for url in url_berisiko]
    + [{"url": url, "skenario": "netral", "expected": "Tidak Dipaksa"} for url in url_netral]
)

lokasi_input_validasi_url = direktori_examples / "input_url_step17_validasi_engine_v5.csv"
data_skenario_url.to_csv(lokasi_input_validasi_url, index=False, encoding="utf-8")

print("Dataset skenario URL siap:", lokasi_input_validasi_url)
display(data_skenario_url.head(20))


Dataset skenario URL siap: C:\Users\ASUS\PHISHING\examples\input_url_step17_validasi_engine_v5.csv


,url,skenario,expected
0,https://praktikum.gunadarma.ac.id,resmi,Terlihat Aman
1,https://baak.gunadarma.ac.id,resmi,Terlihat Aman
2,https://www.bca.co.id,resmi,Terlihat Aman
3,https://www.shopee.co.id,resmi,Terlihat Aman
4,https://www.microsoft.com,resmi,Terlihat Aman
5,https://google.com,resmi,Terlihat Aman
6,https://youtube.com,resmi,Terlihat Aman
7,https://apple.com,resmi,Terlihat Aman
8,https://cloudflare.com,resmi,Terlihat Aman
9,https://github.com,resmi,Terlihat Aman


In [4]:
def analisis_skenario_url(engine, data_skenario, nama_mode):
    hasil = []

    for _, row in data_skenario.iterrows():
        item = engine.analisis_url(row["url"])
        item["skenario"] = row["skenario"]
        item["expected"] = row["expected"]
        item["mode_model"] = nama_mode
        hasil.append(item)

    return pd.DataFrame(hasil)


hasil_url_best = analisis_skenario_url(engine_best, data_skenario_url, "best")
lokasi_hasil_url_best = direktori_outputs / "hasil_validasi_url_engine_v5_best_step17.csv"
hasil_url_best.to_csv(lokasi_hasil_url_best, index=False, encoding="utf-8")

kolom_tampil = [
    "url",
    "skenario",
    "expected",
    "skor_model_v5",
    "skor_final_v5",
    "kategori_risiko_v5",
    "hasil_akhir_v5",
    "intelligence_status",
    "public_ti_status",
    "alasan_v5",
]

kolom_tampil = [kolom for kolom in kolom_tampil if kolom in hasil_url_best.columns]

print("Validasi URL mode best selesai:", lokasi_hasil_url_best)
display(hasil_url_best[kolom_tampil])


Validasi URL mode best selesai: C:\Users\ASUS\PHISHING\reports\outputs\hasil_validasi_url_engine_v5_best_step17.csv


,url,skenario,expected,skor_model_v5,skor_final_v5,kategori_risiko_v5,hasil_akhir_v5,intelligence_status,public_ti_status,alasan_v5
0,https://praktikum.gunadarma.ac.id,resmi,Terlihat Aman,68.49,24.00,Rendah,Terlihat Aman,resmi_terlihat_aman,tidak_ditemukan_di_public_ti,Domain cocok dengan daftar resmi dan tidak pun...
1,https://baak.gunadarma.ac.id,resmi,Terlihat Aman,32.91,24.00,Rendah,Terlihat Aman,resmi_terlihat_aman,tidak_ditemukan_di_public_ti,Domain cocok dengan daftar resmi dan tidak pun...
2,https://www.bca.co.id,resmi,Terlihat Aman,7.99,7.99,Rendah,Terlihat Aman,resmi_terlihat_aman,tidak_ditemukan_di_public_ti,Domain cocok dengan daftar resmi dan tidak pun...
3,https://www.shopee.co.id,resmi,Terlihat Aman,7.58,7.58,Rendah,Terlihat Aman,resmi_terlihat_aman,tidak_ditemukan_di_public_ti,Domain cocok dengan daftar resmi dan tidak pun...
4,https://www.microsoft.com,resmi,Terlihat Aman,3.63,3.63,Rendah,Terlihat Aman,resmi_terlihat_aman,tidak_ditemukan_di_public_ti,Domain cocok dengan daftar resmi dan tidak pun...
5,https://google.com,resmi,Terlihat Aman,2.33,2.33,Rendah,Terlihat Aman,resmi_terlihat_aman,tidak_ditemukan_di_public_ti,Domain cocok dengan daftar resmi dan tidak pun...
6,https://youtube.com,resmi,Terlihat Aman,1.04,1.04,Rendah,Terlihat Aman,belum_ada_sinyal_kuat,tidak_ditemukan_di_public_ti,Keputusan berdasarkan skor model V5 dan sinyal...
7,https://apple.com,resmi,Terlihat Aman,4.00,4.00,Rendah,Terlihat Aman,resmi_terlihat_aman,tidak_ditemukan_di_public_ti,Domain cocok dengan daftar resmi dan tidak pun...
8,https://cloudflare.com,resmi,Terlihat Aman,2.35,2.35,Rendah,Terlihat Aman,resmi_terlihat_aman,tidak_ditemukan_di_public_ti,Domain cocok dengan daftar resmi dan tidak pun...
9,https://github.com,resmi,Terlihat Aman,2.33,2.33,Rendah,Terlihat Aman,resmi_terlihat_aman,tidak_ditemukan_di_public_ti,Domain cocok dengan daftar resmi dan tidak pun...


In [5]:
def nilai_status_validasi(row):
    skenario = row.get("skenario", "")
    hasil = row.get("hasil_akhir_v5", "")

    if skenario == "resmi":
        return "lolos" if hasil in ["Terlihat Aman", "Perlu Tinjauan"] else "gagal"

    if skenario == "berisiko":
        return "lolos" if hasil == "Berisiko" else "gagal"

    return "observasi"


hasil_url_best["status_validasi"] = hasil_url_best.apply(nilai_status_validasi, axis=1)

ringkasan_validasi_url = (
    hasil_url_best
    .groupby(["skenario", "status_validasi"])
    .size()
    .reset_index(name="jumlah_data")
)

temuan_gagal_url = hasil_url_best[
    hasil_url_best["status_validasi"] == "gagal"
].copy()

lokasi_ringkasan_validasi_url = direktori_outputs / "ringkasan_validasi_url_engine_v5_step17.csv"
lokasi_temuan_gagal_url = direktori_outputs / "temuan_gagal_validasi_url_engine_v5_step17.csv"

ringkasan_validasi_url.to_csv(lokasi_ringkasan_validasi_url, index=False, encoding="utf-8")
temuan_gagal_url.to_csv(lokasi_temuan_gagal_url, index=False, encoding="utf-8")

print("Ringkasan validasi URL:")
display(ringkasan_validasi_url)

print("Temuan gagal:")
display(temuan_gagal_url[kolom_tampil].head(30))


Ringkasan validasi URL:


,skenario,status_validasi,jumlah_data
0,berisiko,lolos,15
1,netral,observasi,8
2,resmi,lolos,15


Temuan gagal:


,url,skenario,expected,skor_model_v5,skor_final_v5,kategori_risiko_v5,hasil_akhir_v5,intelligence_status,public_ti_status,alasan_v5


In [6]:
hasil_url_xgb = analisis_skenario_url(engine_xgb, data_skenario_url, "xgb")

kolom_band = [
    "url",
    "skenario",
    "expected",
    "skor_model_v5",
    "skor_final_v5",
    "kategori_risiko_v5",
    "hasil_akhir_v5",
]

band_best = hasil_url_best[kolom_band].copy()
band_xgb = hasil_url_xgb[kolom_band].copy()

band_best = band_best.rename(columns={
    "skor_model_v5": "skor_model_best",
    "skor_final_v5": "skor_final_best",
    "kategori_risiko_v5": "kategori_best",
    "hasil_akhir_v5": "hasil_best",
})

band_xgb = band_xgb.rename(columns={
    "skor_model_v5": "skor_model_xgb",
    "skor_final_v5": "skor_final_xgb",
    "kategori_risiko_v5": "kategori_xgb",
    "hasil_akhir_v5": "hasil_xgb",
})

data_perbandingan_model_v5 = band_best.merge(
    band_xgb[["url", "skor_model_xgb", "skor_final_xgb", "kategori_xgb", "hasil_xgb"]],
    on="url",
    how="left",
)

data_perbandingan_model_v5["hasil_berbeda"] = (
    data_perbandingan_model_v5["hasil_best"] != data_perbandingan_model_v5["hasil_xgb"]
)

lokasi_perbandingan_mode_model_v5 = direktori_outputs / "perbandingan_mode_best_xgb_engine_v5_step17.csv"
data_perbandingan_model_v5.to_csv(lokasi_perbandingan_mode_model_v5, index=False, encoding="utf-8")

print("Perbandingan mode best dan xgb selesai:", lokasi_perbandingan_mode_model_v5)
display(data_perbandingan_model_v5)


Perbandingan mode best dan xgb selesai: C:\Users\ASUS\PHISHING\reports\outputs\perbandingan_mode_best_xgb_engine_v5_step17.csv


,url,skenario,expected,skor_model_best,skor_final_best,kategori_best,hasil_best,skor_model_xgb,skor_final_xgb,kategori_xgb,hasil_xgb,hasil_berbeda
0,https://praktikum.gunadarma.ac.id,resmi,Terlihat Aman,68.49,24.00,Rendah,Terlihat Aman,71.91,24.00,Rendah,Terlihat Aman,False
1,https://baak.gunadarma.ac.id,resmi,Terlihat Aman,32.91,24.00,Rendah,Terlihat Aman,68.91,24.00,Rendah,Terlihat Aman,False
2,https://www.bca.co.id,resmi,Terlihat Aman,7.99,7.99,Rendah,Terlihat Aman,0.90,0.90,Rendah,Terlihat Aman,False
3,https://www.shopee.co.id,resmi,Terlihat Aman,7.58,7.58,Rendah,Terlihat Aman,0.54,0.54,Rendah,Terlihat Aman,False
4,https://www.microsoft.com,resmi,Terlihat Aman,3.63,3.63,Rendah,Terlihat Aman,0.36,0.36,Rendah,Terlihat Aman,False
5,https://google.com,resmi,Terlihat Aman,2.33,2.33,Rendah,Terlihat Aman,0.17,0.17,Rendah,Terlihat Aman,False
6,https://youtube.com,resmi,Terlihat Aman,1.04,1.04,Rendah,Terlihat Aman,1.10,1.10,Rendah,Terlihat Aman,False
7,https://apple.com,resmi,Terlihat Aman,4.00,4.00,Rendah,Terlihat Aman,0.11,0.11,Rendah,Terlihat Aman,False
8,https://cloudflare.com,resmi,Terlihat Aman,2.35,2.35,Rendah,Terlihat Aman,0.23,0.23,Rendah,Terlihat Aman,False
9,https://github.com,resmi,Terlihat Aman,2.33,2.33,Rendah,Terlihat Aman,0.17,0.17,Rendah,Terlihat Aman,False


In [7]:
def ekstrak_url_dari_teks(teks):
    if not isinstance(teks, str):
        teks = str(teks)

    pola = re.compile(r"https?://[^\s<>'\"\\)\]\}]+", flags=re.IGNORECASE)
    hasil = []

    for match in pola.findall(teks):
        url = match.strip().rstrip(".,;:!?\"'<>)]}")
        if url and url not in hasil:
            hasil.append(url)

    return hasil


def ekstrak_url_dari_file_aman(lokasi_file, batas_baca=2_000_000):
    lokasi_file = Path(lokasi_file)
    daftar_url = []

    try:
        if lokasi_file.suffix.lower() == ".zip":
            with zipfile.ZipFile(lokasi_file, "r") as z:
                for nama in z.namelist()[:30]:
                    if nama.endswith("/"):
                        continue

                    try:
                        data = z.read(nama)[:batas_baca]
                        teks = data.decode("utf-8", errors="ignore")
                        daftar_url.extend(ekstrak_url_dari_teks(teks))
                    except Exception:
                        continue
        else:
            data = lokasi_file.read_bytes()[:batas_baca]
            teks = data.decode("utf-8", errors="ignore")
            daftar_url.extend(ekstrak_url_dari_teks(teks))
    except Exception:
        return []

    hasil_unik = []
    for url in daftar_url:
        if url not in hasil_unik:
            hasil_unik.append(url)

    return hasil_unik


if direktori_samples.exists():
    daftar_file_sample = [file for file in direktori_samples.iterdir() if file.is_file()]
else:
    daftar_file_sample = []

hasil_validasi_file = []
hasil_url_dalam_file = []

for file in daftar_file_sample:
    hasil_file = engine_best.analisis_file(file)
    hasil_file["nama_file"] = file.name
    hasil_validasi_file.append(hasil_file)

    daftar_url_file = ekstrak_url_dari_file_aman(file)

    for url in daftar_url_file:
        hasil_url = engine_best.analisis_url(url)
        hasil_url["nama_file_sumber"] = file.name
        hasil_url_dalam_file.append(hasil_url)

data_validasi_file = pd.DataFrame(hasil_validasi_file)
data_url_dalam_file = pd.DataFrame(hasil_url_dalam_file)

lokasi_validasi_file_v5 = direktori_outputs / "hasil_validasi_file_engine_v5_step17.csv"
lokasi_url_dalam_file_v5 = direktori_outputs / "hasil_validasi_url_dalam_file_engine_v5_step17.csv"

data_validasi_file.to_csv(lokasi_validasi_file_v5, index=False, encoding="utf-8")
data_url_dalam_file.to_csv(lokasi_url_dalam_file_v5, index=False, encoding="utf-8")

print("Validasi file selesai:", lokasi_validasi_file_v5)
print("Validasi URL dalam file selesai:", lokasi_url_dalam_file_v5)

if not data_validasi_file.empty:
    kolom_file = [
        kolom for kolom in [
            "nama_file",
            "ekstensi",
            "hasil_akhir_file_v5",
            "kategori_final_file_v5",
            "skor_final_file_v5",
            "engine_version",
            "rekomendasi_final_file_v5",
        ]
        if kolom in data_validasi_file.columns
    ]
    display(data_validasi_file[kolom_file])

if not data_url_dalam_file.empty:
    kolom_url_file = [
        kolom for kolom in [
            "nama_file_sumber",
            "url",
            "domain",
            "skor_final_v5",
            "kategori_risiko_v5",
            "hasil_akhir_v5",
            "intelligence_status",
        ]
        if kolom in data_url_dalam_file.columns
    ]
    display(data_url_dalam_file[kolom_url_file])
else:
    print("Tidak ada URL yang berhasil diekstrak dari file sample.")


Validasi file selesai: C:\Users\ASUS\PHISHING\reports\outputs\hasil_validasi_file_engine_v5_step17.csv
Validasi URL dalam file selesai: C:\Users\ASUS\PHISHING\reports\outputs\hasil_validasi_url_dalam_file_engine_v5_step17.csv


,nama_file,ekstensi,hasil_akhir_file_v5,kategori_final_file_v5,skor_final_file_v5,engine_version,rekomendasi_final_file_v5
0,contoh_aplikasi_dummy.apk,.apk,Berisiko,Sangat Tinggi,100,V5,File berisiko. Jangan dibuka atau dijalankan s...
1,contoh_arsip_mencurigakan.zip,.zip,Berisiko,Sangat Tinggi,89,V5,File berisiko. Jangan dibuka atau dijalankan s...
2,contoh_catatan_aman.txt,.txt,Terlihat Aman,Rendah,12,V5,File terlihat rendah risiko berdasarkan pemeri...
3,contoh_dokumen_link.docx,.docx,Berisiko,Sangat Tinggi,93,V5,File berisiko. Jangan dibuka atau dijalankan s...
4,contoh_halaman_login.html,.html,Berisiko,Sangat Tinggi,80,V5,File berisiko. Jangan dibuka atau dijalankan s...
5,contoh_pdf_link.pdf,.pdf,Berisiko,Sangat Tinggi,100,V5,File berisiko. Jangan dibuka atau dijalankan s...
6,contoh_pesan_mencurigakan.txt,.txt,Berisiko,Sangat Tinggi,80,V5,File berisiko. Jangan dibuka atau dijalankan s...


,nama_file_sumber,url,domain,skor_final_v5,kategori_risiko_v5,hasil_akhir_v5,intelligence_status
0,contoh_aplikasi_dummy.apk,http://ovo-login-update.test,ovo-login-update.test,92.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko
1,contoh_arsip_mencurigakan.zip,http://paypal-verify-account.test,paypal-verify-account.test,92.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko
2,contoh_catatan_aman.txt,https://praktikum.gunadarma.ac.id,praktikum.gunadarma.ac.id,24.00,Rendah,Terlihat Aman,resmi_terlihat_aman
3,contoh_dokumen_link.docx,http://shopee-login-update.test,shopee-login-update.test,92.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko
4,contoh_dokumen_link.docx,http://bca-login-update.test,bca-login-update.test,92.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko
5,contoh_halaman_login.html,http://paypal-verify-account.test/login,paypal-verify-account.test,94.18,Sangat Tinggi,Berisiko,tiruan_brand_berisiko
6,contoh_pdf_link.pdf,https://micros0ft-login-update.test,micros0ft-login-update.test,92.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko
7,contoh_pesan_mencurigakan.txt,http://bca-login-update.test,bca-login-update.test,92.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko


In [8]:
input_cli_step17 = direktori_examples / "input_url_step17_cli_engine_v5.csv"
hasil_cli_step17 = direktori_outputs / "hasil_cli_step17_engine_v5.csv"

data_skenario_url[["url"]].to_csv(input_cli_step17, index=False, encoding="utf-8")

perintah_cli = [
    sys.executable,
    str(direktori_src / "run_phishrisk_v5.py"),
    "--mode",
    "urls",
    "--input",
    str(input_cli_step17),
    "--url-column",
    "url",
    "--output",
    str(hasil_cli_step17),
    "--model-mode",
    "best",
]

hasil_cli = subprocess.run(
    perintah_cli,
    capture_output=True,
    text=True,
)

print("Return code:", hasil_cli.returncode)
print("STDOUT:")
print(hasil_cli.stdout)
print("STDERR:")
print(hasil_cli.stderr)

if hasil_cli.returncode != 0:
    raise RuntimeError("CLI V5 gagal pada STEP 17.")

data_cli_step17 = pd.read_csv(hasil_cli_step17)
print("Output CLI:", hasil_cli_step17)
display(data_cli_step17.head(20))


Return code: 0
STDOUT:
PhishRisk Engine V5 selesai.
Mode: urls
Output: C:\Users\ASUS\PHISHING\reports\outputs\hasil_cli_step17_engine_v5.csv
hasil_akhir_v5 kategori_risiko_v5  skor_final_v5 engine_version
 Terlihat Aman             Rendah          24.00             V5
 Terlihat Aman             Rendah          24.00             V5
 Terlihat Aman             Rendah           7.99             V5
 Terlihat Aman             Rendah           7.58             V5
 Terlihat Aman             Rendah           3.63             V5
 Terlihat Aman             Rendah           2.33             V5
 Terlihat Aman             Rendah           1.04             V5
 Terlihat Aman             Rendah           4.00             V5
 Terlihat Aman             Rendah           2.35             V5
 Terlihat Aman             Rendah           2.33             V5

STDERR:

Output CLI: C:\Users\ASUS\PHISHING\reports\outputs\hasil_cli_step17_engine_v5.csv


,url,domain,tld,probabilitas_model,skor_model,label_model,skor_final,kategori_risiko,hasil_akhir,rekomendasi,...,brand_keyword_detected,suspicious_keyword_count,skor_final_v5,kategori_risiko_v5,hasil_akhir_v5,alasan_v5,rekomendasi_v5,engine_version,model_path_v5,threshold_referensi_v5
0,https://praktikum.gunadarma.ac.id,praktikum.gunadarma.ac.id,id,0.204000,20.40,Legitimate,20.40,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,...,1,0,24.00,Rendah,Terlihat Aman,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...,V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
1,https://baak.gunadarma.ac.id,baak.gunadarma.ac.id,id,0.316000,31.60,Legitimate,24.00,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,...,1,0,24.00,Rendah,Terlihat Aman,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...,V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
2,https://www.bca.co.id,bca.co.id,id,0.040543,4.05,Legitimate,4.05,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,...,1,0,7.99,Rendah,Terlihat Aman,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...,V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
3,https://www.shopee.co.id,shopee.co.id,id,0.092183,9.22,Legitimate,9.22,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,...,1,0,7.58,Rendah,Terlihat Aman,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...,V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
4,https://www.microsoft.com,microsoft.com,com,0.480013,48.00,Legitimate,24.00,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,...,1,0,3.63,Rendah,Terlihat Aman,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...,V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
5,https://google.com,google.com,com,0.020000,2.00,Legitimate,2.00,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,...,1,0,2.33,Rendah,Terlihat Aman,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...,V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
6,https://youtube.com,youtube.com,com,0.948000,94.80,Phishing,94.80,Sangat Tinggi,Berisiko,"Alamat berisiko. Jangan dibuka, jangan diisi, ...",...,0,0,1.04,Rendah,Terlihat Aman,Keputusan berdasarkan skor model V5 dan sinyal...,Alamat terlihat rendah risiko. Tetap pastikan ...,V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
7,https://apple.com,apple.com,com,0.056000,5.60,Legitimate,5.60,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,...,1,0,4.00,Rendah,Terlihat Aman,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...,V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
8,https://cloudflare.com,cloudflare.com,com,0.156000,15.60,Legitimate,15.60,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,...,1,0,2.35,Rendah,Terlihat Aman,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...,V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15
9,https://github.com,github.com,com,0.020000,2.00,Legitimate,2.00,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,...,1,0,2.33,Rendah,Terlihat Aman,Domain cocok dengan daftar resmi dan tidak pun...,Alamat terlihat rendah risiko. Tetap pastikan ...,V5,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,0.15


In [9]:
jumlah_benchmark = min(30, len(data_skenario_url))
url_benchmark = data_skenario_url["url"].head(jumlah_benchmark).tolist()

waktu_mulai = time.perf_counter()
_ = engine_best.analisis_banyak_url(url_benchmark)
durasi_total = time.perf_counter() - waktu_mulai

data_benchmark = pd.DataFrame([{
    "jumlah_url": jumlah_benchmark,
    "durasi_detik": round(durasi_total, 4),
    "rata_rata_detik_per_url": round(durasi_total / jumlah_benchmark, 4) if jumlah_benchmark else 0,
    "model_path": str(engine_best.model_path),
}])

lokasi_benchmark_v5 = direktori_outputs / "benchmark_ringkas_engine_v5_step17.csv"
data_benchmark.to_csv(lokasi_benchmark_v5, index=False, encoding="utf-8")

print("Benchmark ringkas selesai:", lokasi_benchmark_v5)
display(data_benchmark)


Benchmark ringkas selesai: C:\Users\ASUS\PHISHING\reports\outputs\benchmark_ringkas_engine_v5_step17.csv


,jumlah_url,durasi_detik,rata_rata_detik_per_url,model_path
0,30,18.2413,0.608,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...


In [10]:
ringkasan_status = {
    "jumlah_url_uji": int(len(hasil_url_best)),
    "jumlah_validasi_gagal": int(len(temuan_gagal_url)),
    "jumlah_file_uji": int(len(data_validasi_file)) if "data_validasi_file" in globals() else 0,
    "jumlah_url_dalam_file": int(len(data_url_dalam_file)) if "data_url_dalam_file" in globals() else 0,
    "cli_return_code": int(hasil_cli.returncode),
    "mode_best_model": str(engine_best.model_path),
    "mode_xgb_model": str(engine_xgb.model_path),
}

siap_streamlit = (
    ringkasan_status["jumlah_validasi_gagal"] == 0
    and ringkasan_status["cli_return_code"] == 0
    and ringkasan_status["jumlah_file_uji"] > 0
)

status_engine = "siap_uji_streamlit" if siap_streamlit else "perlu_tinjauan_lanjutan"

data_status_step17 = pd.DataFrame([{
    **ringkasan_status,
    "status_engine_v5": status_engine,
    "catatan": (
        "Engine V5 siap diuji pada Streamlit."
        if siap_streamlit
        else "Masih ada bagian yang perlu ditinjau sebelum Streamlit memakai Engine V5."
    ),
}])

lokasi_status_step17 = direktori_outputs / "status_kesiapan_engine_v5_step17.csv"
data_status_step17.to_csv(lokasi_status_step17, index=False, encoding="utf-8")

print("Status kesiapan Engine V5:")
display(data_status_step17)


Status kesiapan Engine V5:


,jumlah_url_uji,jumlah_validasi_gagal,jumlah_file_uji,jumlah_url_dalam_file,cli_return_code,mode_best_model,mode_xgb_model,status_engine_v5,catatan
0,38,0,7,8,0,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...,C:\Users\ASUS\PHISHING\models\model_xgb_multi_...,siap_uji_streamlit,Engine V5 siap diuji pada Streamlit.


In [11]:
file_output_step17 = [
    lokasi_hasil_url_best,
    lokasi_ringkasan_validasi_url,
    lokasi_temuan_gagal_url,
    lokasi_perbandingan_mode_model_v5,
    lokasi_validasi_file_v5,
    lokasi_url_dalam_file_v5,
    hasil_cli_step17,
    lokasi_benchmark_v5,
    lokasi_status_step17,
]

data_validasi_output_step17 = []

for lokasi in file_output_step17:
    lokasi = Path(lokasi)
    data_validasi_output_step17.append({
        "nama_file": lokasi.name,
        "lokasi": str(lokasi),
        "tersedia": lokasi.exists(),
        "ukuran_kb": round(lokasi.stat().st_size / 1024, 2) if lokasi.exists() else 0,
    })

data_validasi_output_step17 = pd.DataFrame(data_validasi_output_step17)

lokasi_validasi_output_step17 = direktori_outputs / "validasi_step17_engine_v5.csv"
data_validasi_output_step17.to_csv(lokasi_validasi_output_step17, index=False, encoding="utf-8")

metadata_step17 = {
    "nama_notebook": "17_validasi_lanjutan_engine_v5.ipynb",
    "nama_tahap": "Validasi Lanjutan Engine V5",
    "status": status_engine,
    "ringkasan_status": ringkasan_status,
    "file_output": {
        "hasil_url_best": str(lokasi_hasil_url_best),
        "ringkasan_validasi_url": str(lokasi_ringkasan_validasi_url),
        "temuan_gagal_url": str(lokasi_temuan_gagal_url),
        "perbandingan_model": str(lokasi_perbandingan_mode_model_v5),
        "validasi_file": str(lokasi_validasi_file_v5),
        "url_dalam_file": str(lokasi_url_dalam_file_v5),
        "hasil_cli": str(hasil_cli_step17),
        "benchmark": str(lokasi_benchmark_v5),
        "status": str(lokasi_status_step17),
        "validasi_output": str(lokasi_validasi_output_step17),
    },
    "catatan": [
        "Validasi URL resmi dan berisiko memakai skenario eksplisit.",
        "Mode best dan xgb dibandingkan untuk melihat opsi deployment ringan.",
        "URL dalam file diekstrak memakai fallback regex aman agar hasil tidak kosong.",
        "Jika status siap_uji_streamlit, Engine V5 boleh mulai diintegrasikan ke website untuk pengujian.",
    ],
    "tanggal_selesai": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

lokasi_metadata_step17 = direktori_outputs / "metadata_step17_validasi_engine_v5.json"
lokasi_metadata_step17.write_text(
    json.dumps(metadata_step17, indent=4, ensure_ascii=False),
    encoding="utf-8",
)

catatan_step17 = f"""
CATATAN FINAL STEP 17

Notebook:
17_validasi_lanjutan_engine_v5.ipynb

Status:
{status_engine}

Output utama:
{lokasi_status_step17}

Catatan:
Validasi lanjutan Engine V5 sudah mencakup URL resmi, URL berisiko, mode model, file analyzer, URL dalam file, CLI, dan benchmark ringkas.

Tahap berikutnya:
18_integrasi_streamlit_engine_v5.ipynb
"""

lokasi_catatan_step17 = direktori_outputs / "catatan_final_step17_validasi_engine_v5.txt"
lokasi_catatan_step17.write_text(catatan_step17, encoding="utf-8")

print(catatan_step17)
print("Metadata:", lokasi_metadata_step17)
print("Validasi output:", lokasi_validasi_output_step17)

display(data_validasi_output_step17)



CATATAN FINAL STEP 17

Notebook:
17_validasi_lanjutan_engine_v5.ipynb

Status:
siap_uji_streamlit

Output utama:
C:\Users\ASUS\PHISHING\reports\outputs\status_kesiapan_engine_v5_step17.csv

Catatan:
Validasi lanjutan Engine V5 sudah mencakup URL resmi, URL berisiko, mode model, file analyzer, URL dalam file, CLI, dan benchmark ringkas.

Tahap berikutnya:
18_integrasi_streamlit_engine_v5.ipynb

Metadata: C:\Users\ASUS\PHISHING\reports\outputs\metadata_step17_validasi_engine_v5.json
Validasi output: C:\Users\ASUS\PHISHING\reports\outputs\validasi_step17_engine_v5.csv


,nama_file,lokasi,tersedia,ukuran_kb
0,hasil_validasi_url_engine_v5_best_step17.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_v...,True,58.98
1,ringkasan_validasi_url_engine_v5_step17.csv,C:\Users\ASUS\PHISHING\reports\outputs\ringkas...,True,0.09
2,temuan_gagal_validasi_url_engine_v5_step17.csv,C:\Users\ASUS\PHISHING\reports\outputs\temuan_...,True,1.63
3,perbandingan_mode_best_xgb_engine_v5_step17.csv,C:\Users\ASUS\PHISHING\reports\outputs\perband...,True,4.52
4,hasil_validasi_file_engine_v5_step17.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_v...,True,8.56
5,hasil_validasi_url_dalam_file_engine_v5_step17...,C:\Users\ASUS\PHISHING\reports\outputs\hasil_v...,True,15.26
6,hasil_cli_step17_engine_v5.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_c...,True,58.04
7,benchmark_ringkas_engine_v5_step17.csv,C:\Users\ASUS\PHISHING\reports\outputs\benchma...,True,0.14
8,status_kesiapan_engine_v5_step17.csv,C:\Users\ASUS\PHISHING\reports\outputs\status_...,True,0.33


## 